# Apache Iceberg's Dark Logic
## Building a Context Graph to Make Institutional Memory Machine-Readable

Foundation Capital's essay [*"Context Graphs: AI's Trillion-Dollar Opportunity"*](https://foundationcapital.com/ideas/context-graphs-ais-trillion-dollar-opportunity)
identifies a structural gap: traditional systems of record capture *what happened* but discard *why*.
Exception logic, trade-offs, and precedents — the "dark logic" — live only in scattered conversations
and tribal knowledge. AI agents can't reason over it.

This notebook demonstrates a concrete proof of that thesis using Apache Iceberg's GitHub history.

**What we built:**
- Scraped 200 merged PRs + 200 closed issues from [apache/iceberg](https://github.com/apache/iceberg)
- Used Claude to extract `LogicNodes` — named decision traces capturing the *Why* behind each change
- Loaded everything into a Neo4j **Context Graph**
- Built a two-stage reasoning agent: `question → Cypher → answer`

**This is graph traversal, not RAG.** Results are exact and cited, not approximately similar.

**Graph schema:**
```
(Actor)-[:PROPOSED]->(DecisionEvent)-[:ESTABLISHES]->(LogicNode)-[:APPLIES_TO]->(Artifact)
         [:VALIDATED]                [:SUPERSEDES]
                                      [:REFERENCES]->(DecisionEvent)
                                      [:LABELLED]->(Label)
```

---

In [8]:
import sys
sys.path.insert(0, "..")

from src.agent import ask
from src.config import get_neo4j_driver
import json

def pretty(result: dict):
    """Print a query result in a readable format for the article."""
    print(f"{'─'*70}")
    print(f"QUESTION:\n  {result['question']}")
    print(f"\nCYPHER GENERATED:\n  {result['cypher']}")
    print(f"\nRAW RESULTS ({len(result['raw_results'])} rows):")
    for row in result['raw_results'][:8]:
        print(f"  {row}")
    print(f"\nANSWER:\n{result['answer']}")
    print(f"{'─'*70}\n")

## Graph Statistics
Let's first check what we ingested.

In [2]:
driver = get_neo4j_driver()
with driver.session() as session:
    node_counts = session.run(
        "MATCH (n) RETURN labels(n)[0] AS label, count(n) AS count ORDER BY count DESC"
    ).data()
    rel_counts = session.run(
        "MATCH ()-[r]->() RETURN type(r) AS rel, count(r) AS count ORDER BY count DESC"
    ).data()
driver.close()

print("Node counts:")
for row in node_counts:
    print(f"  {row['label']:<20} {row['count']:>6}")
print("\nRelationship counts:")
for row in rel_counts:
    print(f"  {row['rel']:<20} {row['count']:>6}")

Node counts:
  DecisionEvent           538
  Artifact                511
  LogicNode               332
  Actor                   158
  Label                    31

Relationship counts:
  APPLIES_TO             1303
  LABELLED                484
  VALIDATED               381
  ESTABLISHES             340
  PROPOSED                305
  REFERENCES              289


---
## Query 1: Expertise Map

> **"Who are the domain experts for partition spec decisions?"**

Traditional search: grep for "partition" across PRs and hope for the best.  
Context graph: traverse `VALIDATED → DecisionEvent → ESTABLISHES → LogicNode` filtered by domain,
count decisions per actor. The result is an expertise map derived from actual decision history —
not self-reported skills or org charts.

In [3]:
result = ask("Who are the domain experts for partition spec decisions? Show me the top contributors.")
pretty(result)

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `SUPERSEDES` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=72, offset=71>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 71, 'line': 1, 'column': 72}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (a:Actor)-[:PROPOSED|VALIDATED]->(d:DecisionEvent)-[:ESTABLISHES|SUPERSEDES]->(l:LogicNode {domain: 'partitioning'})\nWITH a, COUNT(DISTINCT d) AS contributions\nORDER BY contributions DESC\nLIMIT 20\nRETURN a.login AS expert, contributions"


──────────────────────────────────────────────────────────────────────
QUESTION:
  Who are the domain experts for partition spec decisions? Show me the top contributors.

CYPHER GENERATED:
  MATCH (a:Actor)-[:PROPOSED|VALIDATED]->(d:DecisionEvent)-[:ESTABLISHES|SUPERSEDES]->(l:LogicNode {domain: 'partitioning'})
WITH a, COUNT(DISTINCT d) AS contributions
ORDER BY contributions DESC
LIMIT 20
RETURN a.login AS expert, contributions

RAW RESULTS (17 rows):
  {'expert': 'RussellSpitzer', 'contributions': 2}
  {'expert': 'huaxingao', 'contributions': 2}
  {'expert': 'singhpk234', 'contributions': 2}
  {'expert': 'lirui-apache', 'contributions': 2}
  {'expert': 'dramaticlly', 'contributions': 1}
  {'expert': 'fb913bf0de288ba84fe98f7a23d35edfdb22381', 'contributions': 1}
  {'expert': 'rdblue', 'contributions': 1}
  {'expert': 'jackylee-ch', 'contributions': 1}

ANSWER:
## Domain Experts: Partition Spec Decisions

### Top Contributors

The graph reveals **17 distinct actors** who have proposed

---
## Query 2: Decision Traces — Surfacing the Dark Logic

> **"What architectural rules govern delete file handling in Iceberg?"**

The *Why* behind delete file design is buried across dozens of PRs. No single document captures
it. The graph surfaces it as a set of named `LogicNode` objects — each with a rationale, the PR
that established it, and the components it applies to.

In [4]:
result = ask("What are the key architectural rules and trade-offs governing delete file handling in Iceberg? Show the logic nodes and the PRs that established them.")
pretty(result)

──────────────────────────────────────────────────────────────────────
QUESTION:
  What are the key architectural rules and trade-offs governing delete file handling in Iceberg? Show the logic nodes and the PRs that established them.

CYPHER GENERATED:
  MATCH (d:DecisionEvent)-[:ESTABLISHES]->(l:LogicNode)
WHERE l.domain = 'delete-files'
OPTIONAL MATCH (a:Actor)-[:PROPOSED]->(d)
OPTIONAL MATCH (d)-[:LABELLED]->(lbl:Label)
OPTIONAL MATCH (l)-[:APPLIES_TO]->(art:Artifact)
RETURN
    l.name                AS logic_node,
    l.description         AS rule_description,
    collect(DISTINCT d.id)        AS establishing_prs,
    collect(DISTINCT d.title)     AS pr_titles,
    collect(DISTINCT d.date)      AS pr_dates,
    collect(DISTINCT a.login)     AS authors,
    collect(DISTINCT art.name)    AS affected_artifacts,
    collect(DISTINCT lbl.name)    AS labels
ORDER BY l.name
LIMIT 20

RAW RESULTS (20 rows):
  {'logic_node': '64-bit range split at key boundaries for native API delegation', 

---
## Query 3: Impact Analysis

> **"If I want to change the partition spec, what other components and domains will be affected?"**

A SQL table of PRs can tell you which files changed. It can't tell you which *other areas of the
codebase carry decisions that depend on the partition spec*. This query follows the graph:
`LogicNode {domain: partitioning} → APPLIES_TO → Artifact → back to other DecisionEvents` —
surfacing cross-cutting impact that would otherwise require reading hundreds of PRs manually.

In [9]:
result = ask("If I want to change the partition spec in Iceberg, which other components and domains have been affected by partition spec decisions in the past? Show the cross-cutting impact.")
pretty(result)

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `SUPERSEDES` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=39, offset=38>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 38, 'line': 1, 'column': 39}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (d:DecisionEvent)-[:ESTABLISHES|SUPERSEDES]->(l:LogicNode {domain: 'partitioning'})\nMATCH (l)-[:APPLIES_TO]->(a:Artifact)\nWITH d, l, a\nMATCH (l2:LogicNode)-[:APPLIES_TO]->(a)\nWHERE l2.domain <> 'partitioning'\nWITH \n  a.name            AS shared_artifact,\n  l.name            AS partiti

──────────────────────────────────────────────────────────────────────
QUESTION:
  If I want to change the partition spec in Iceberg, which other components and domains have been affected by partition spec decisions in the past? Show the cross-cutting impact.

CYPHER GENERATED:
  MATCH (d:DecisionEvent)-[:ESTABLISHES|SUPERSEDES]->(l:LogicNode {domain: 'partitioning'})
MATCH (l)-[:APPLIES_TO]->(a:Artifact)
WITH d, l, a
MATCH (l2:LogicNode)-[:APPLIES_TO]->(a)
WHERE l2.domain <> 'partitioning'
WITH 
  a.name            AS shared_artifact,
  l.name            AS partitioning_rule,
  l2.name           AS cross_domain_rule,
  l2.domain         AS cross_domain,
  d.id              AS decision_id,
  d.title           AS decision_title,
  d.decision_type   AS decision_type,
  d.date            AS date
RETURN 
  shared_artifact,
  cross_domain,
  collect(DISTINCT cross_domain_rule)  AS affected_rules,
  collect(DISTINCT partitioning_rule)  AS partitioning_rules_involved,
  collect(DISTINCT decis

---
## Query 4: Temporal Evolution

> **"How did the delete file strategy evolve over time in Iceberg?"**

Q2 showed the rules that exist *now*. This shows *how they got there* — decisions ordered
chronologically, revealing which rules were established early, which were corrections to earlier
decisions, and which were added as the implementation matured under production load.

In [11]:
result = ask("How did the delete file strategy evolve over time in Iceberg? Show the key decisions in chronological order with their rationale.")
pretty(result)

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `SUPERSEDES` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=40, offset=39>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 39, 'line': 1, 'column': 40}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (de:DecisionEvent)-[:ESTABLISHES|SUPERSEDES]->(ln:LogicNode {domain: 'delete-files'})\nWITH de, ln\nORDER BY de.date ASC\nRETURN\n    de.date          AS decision_date,\n    de.id            AS decision_id,\n    de.title         AS title,\n    de.type          AS type,\n    de.decision_type 

──────────────────────────────────────────────────────────────────────
QUESTION:
  How did the delete file strategy evolve over time in Iceberg? Show the key decisions in chronological order with their rationale.

CYPHER GENERATED:
  MATCH (de:DecisionEvent)-[:ESTABLISHES|SUPERSEDES]->(ln:LogicNode {domain: 'delete-files'})
WITH de, ln
ORDER BY de.date ASC
RETURN
    de.date          AS decision_date,
    de.id            AS decision_id,
    de.title         AS title,
    de.type          AS type,
    de.decision_type AS decision_type,
    de.author        AS author,
    de.rationale_summary AS rationale,
    de.url           AS url,
    collect(ln.name) AS logic_nodes,
    collect(ln.description) AS logic_descriptions
ORDER BY decision_date ASC
LIMIT 50

RAW RESULTS (23 rows):
  {'decision_date': '2024-02-24T00:10:40Z', 'decision_id': 'github-issue-1026', 'title': 'Add an action to rewrite equality deletes as position deletes', 'type': 'Issue', 'decision_type': 'new-feature', 'author'

---
## Query 5: Unresolved Tensions

> **"What trade-offs and deprecations exist in the format-spec domain?"**

Trade-offs are decisions where someone chose Option A over Option B with known caveats.
Deprecations signal forward pressure — a rule being retired. Both represent open questions
that future contributors need to understand before touching this area.

In [7]:
result = ask("What trade-offs and deprecations exist in the format-spec domain? Show the decision events and their rationale.")
pretty(result)

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `SUPERSEDES` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=39, offset=38>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 38, 'line': 1, 'column': 39}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (d:DecisionEvent)-[:ESTABLISHES|SUPERSEDES]->(l:LogicNode)\nWHERE d.decision_type IN ['trade-off', 'deprecation']\n  AND l.domain = 'format-spec'\nRETURN d.id AS decision_id,\n       d.title AS title,\n       d.type AS type,\n       d.date AS date,\n       d.decision_type AS decision_type,\n

──────────────────────────────────────────────────────────────────────
QUESTION:
  What trade-offs and deprecations exist in the format-spec domain? Show the decision events and their rationale.

CYPHER GENERATED:
  MATCH (d:DecisionEvent)-[:ESTABLISHES|SUPERSEDES]->(l:LogicNode)
WHERE d.decision_type IN ['trade-off', 'deprecation']
  AND l.domain = 'format-spec'
RETURN d.id AS decision_id,
       d.title AS title,
       d.type AS type,
       d.date AS date,
       d.decision_type AS decision_type,
       d.rationale_summary AS rationale,
       d.author AS author,
       d.url AS url,
       collect(DISTINCT l.name) AS logic_nodes
ORDER BY d.decision_type, d.date DESC
LIMIT 20

RAW RESULTS (2 rows):
  {'decision_id': 'github-pr-15575', 'title': 'Core, Flink, Spark: Deprecate Manifest read methods which rely manifest Metadata', 'type': 'PR', 'date': '2026-03-11T20:54:20Z', 'decision_type': 'deprecation', 'rationale': 'Deprecate metadata-reliant manifest read methods to prepare for V4

---
## Try Your Own Question

The agent handles any natural-language question about the graph.

In [ ]:
# Change this to anything you're curious about
my_question = "Which contributors are most active across the most domains?"

result = ask(my_question)
pretty(result)

---
## What's Next

This POC covers one data source (GitHub) and one project (Apache Iceberg). The Foundation Capital
essay identifies three paths to value with context graphs — this POC is a proof of the third:
*new systems of record for decision-making that incumbents can't build.*

To extend it:

1. **Add Slack** — Iceberg's public Slack archive is on [linen.dev](https://linen.dev). A `scrapers/slack.py` stub already exists; the normalised schema is identical so `extractor.py` needs no changes. Slack is where the most informal decision traces live.
2. **Add commit history** — link `Commit` nodes to PRs via the GitHub commits API to connect code-level evidence to the decisions that produced it.
3. **Temporal edges** — when a newer `LogicNode` supersedes an older one, add an `EVOLVED_INTO` edge to trace how rules change over time.
4. **Vector + graph hybrid** — add `body_embedding` to `DecisionEvent` for semantic similarity fallback when Cypher queries return empty results.
5. **Streamlit UI** — an interactive chat interface that renders the Cypher query and a graph visualization side-by-side.

**The broader point:** context graphs are not a nice-to-have audit trail. They are the missing
layer between what AI agents can execute and what they can *reason about*. This graph is a small
proof that "dark logic" buried in open-source project history can be made machine-readable —
and that changes what AI agents can do with institutional knowledge.

See [Foundation Capital's essay](https://foundationcapital.com/ideas/context-graphs-ais-trillion-dollar-opportunity) for the full investment thesis.